## Part 1 - Dataset & Preprocessing

In [2]:
# ================================
# 1️. Mount Google Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')

# ================================
# 2️. Import Libraries
# ================================
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

# ================================
# 3️. Load Dataset from Google Drive
# ================================
# Path updated for Google Drive directory structure
file_path = "/content/drive/MyDrive/dataset/Training_Data_Google_Play_reviews_6000.csv"
df = pd.read_csv(file_path)

print("==== Dataset Preview ====")
print(df.head())

print("\n==== Dataset Info ====")
print(df.info())

print("\n==== Missing Values ====")
print(df.isnull().sum())

# ================================
# 4️. Strip whitespace and filter English reviews only
# ================================
df['userLang'] = df['userLang'].str.strip()
df_en = df[df['userLang'] == 'EN'].copy()
print("\n==== English Reviews Count ====")
print(len(df_en))

# ================================
# 5️. Map app_id to default app names
# ================================
app_mapping = {
    'org.telegram.messenger': 'Telegram',
    'com.facebook.orca': 'Facebook Messenger',
    'com.whatsapp': 'WhatsApp',
    'com.viber.voip': 'Viber',
    'com.snapchat.android': 'Snapchat',
    'com.tencent.mm': 'WeChat'
}

df_en['app_name'] = df_en['app_id'].map(app_mapping)
print("\n==== Sample App Names Mapping ====")
print(df_en[['app_id', 'app_name']].head(10))

# ================================
# 6️. Map scores to sentiment
# ================================
def map_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df_en['Sentiment'] = df_en['score'].apply(map_sentiment)
print("\n==== Sentiment Distribution ====")
print(df_en['Sentiment'].value_counts())

Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


==== Dataset Preview ====
                               reviewId           userName  \
0  495266a4-f451-48c3-a844-fb3c07560d55     Foysal Hossain   
1  947fcd83-7a28-403d-b03b-d0bc20f52e0e          S K VERMA   
2  65856211-67ba-4560-84dd-a0055775ed90      Amanuel Abara   
3  cd5ba250-3a26-43b4-a378-77d18f73a503  Vagarangas X Aopi   
4  e8e886b4-d6c6-416b-b0a1-be90320c4024       Shafin islam   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a-/ALV-U...   
1  https://play-lh.googleusercontent.com/a/ACg8oc...   
2  https://play-lh.googleusercontent.com/a/ACg8oc...   
3  https://play-lh.googleusercontent.com/a/ACg8oc...   
4  https://play-lh.googleusercontent.com/a-/ALV-U...   

                      content  score  thumbsUpCount reviewCreatedVersion  \
0  Gett van for no reason 😂😂😂      1              0                  NaN   
1       better' than WhatsApp      4              0                  NaN   
2    That was good app for me      5

In [4]:
# ================================
# 6️. Convert Review Dates to Datetime
# ================================
df_en['review_date'] = pd.to_datetime(df_en['at'], errors='coerce')
print("\n==== Sample Review Dates ====")
print(df_en[['at', 'review_date']].head(10))

# ================================
# 7️. Preprocessing Function with Stemming
# ================================
nltk.download('punkt_tab') # Add this line to download the missing resource
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def preprocess_text_stem(text):
    """Preprocess text with stemming"""
    if pd.isnull(text):
        return ''
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove punctuation but keep emojis and numbers
    text = re.sub(r'[\w\s\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF]', '', text)
    # Tokenization
    words = nltk.word_tokenize(text)
    # Remove stopwords and apply stemming
    words = [ps.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

# ================================
# 8️ Apply Preprocessing
# ================================
df_en['clean_text'] = df_en['content'].apply(preprocess_text_stem)
print("\n==== Sample Preprocessed Reviews ====")
print(df_en[['content', 'clean_text', 'Sentiment', 'app_name']].head(10))

# ================================
# 9️. Save Preprocessed CSV
# ================================
df_final = df_en[['clean_text', 'Sentiment', 'app_name', 'review_date']]\
    .rename(columns={'clean_text':'text', 'Sentiment':'label'})

df_final.to_csv("preprocessed_reviews.csv", index=False)
print("\n==== Preprocessed dataset saved as 'preprocessed_reviews.csv' ====")


==== Sample Review Dates ====
                    at         review_date
0  2023-09-19 15:05:31 2023-09-19 15:05:31
1  2023-09-19 14:59:30 2023-09-19 14:59:30
2  2023-09-19 14:55:06 2023-09-19 14:55:06
3  2023-09-19 14:50:18 2023-09-19 14:50:18
4  2023-09-19 14:48:20 2023-09-19 14:48:20
5  2023-09-19 14:46:54 2023-09-19 14:46:54
6  2023-09-19 14:43:18 2023-09-19 14:43:18
7  2023-09-19 14:42:49 2023-09-19 14:42:49
8  2023-09-19 14:37:57 2023-09-19 14:37:57
9  2023-09-19 14:37:46 2023-09-19 14:37:46


[nltk_data] Downloading package punkt_tab to /root/nltk_data...



==== Sample Preprocessed Reviews ====
                              content clean_text Sentiment  app_name
0          Gett van for no reason 😂😂😂             negative  Telegram
1               better' than WhatsApp          '  positive  Telegram
2            That was good app for me             positive  Telegram
3                        Love the app             positive  Telegram
4                              🕳️🕳️🕳️        ️️️  negative  Telegram
5                            Good app             positive  Telegram
6                            Nice app             positive  Telegram
7  telegram mare call par nai ban rhi             positive  Telegram
8            now how to mute stories?          ?   neutral  Telegram
9                   Good parfom sab 1             positive  Telegram

==== Preprocessed dataset saved as 'preprocessed_reviews.csv' ====


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Part **2**

In [9]:
# ================================================
# Part 2 - Model Training, Evaluation & Backend Export
# ================================================

# ================================================
# 1️. Import Required Libraries
# ================================================
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.pipeline import Pipeline

# Libraries for Transformers
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# ================================================
# 2️. Load and Prepare Preprocessed Data
# ================================================
print("==== Loading Dataset ====")
df_final = pd.read_csv("preprocessed_reviews.csv")

# Handle any missing/empty text values gracefully to avoid vectorizer crashes
df_final['text'] = df_final['text'].fillna('missing_text')
df_final['text'] = df_final['text'].replace('', 'missing_text')

# Map textual labels to integers for classification
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df_final['target'] = df_final['label'].map(label_mapping)

# Run train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    df_final['text'],
    df_final['target'],
    test_size=0.2,
    random_state=42,
    stratify=df_final['target']
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}\n")

# ================================================
# 3️. Feature Extraction (TF-IDF Vectorizer)
# ================================================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ================================================
# 4️. Model Training & Evaluation (Traditional ML)
# ================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB()
}

results = {}
best_acc = 0
best_model_name = None
best_model_obj = None

for name, model in models.items():
    # Train model
    model.fit(X_train_tfidf, y_train)
    # Predict
    preds = model.predict(X_test_tfidf)

    # Calculate Metrics
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, preds)

    results[name] = {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "confusion_matrix": cm,
        "model": model
    }

    print(f"==== {name} Performance ====")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")

    # Track the absolute best traditional model
    if acc > best_acc:
        best_acc = acc
        best_model_name = name
        best_model_obj = model

# ================================================
# 5️. Export Backend for Streamlit Deployment
# ================================================
print(f"🥇 Absolute Best Traditional Model: {best_model_name} ({best_acc:.4f} Accuracy)")

best_pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('classifier', best_model_obj)
])

joblib.dump(best_pipeline, 'best_sentiment_pipeline.pkl')
print("Pipeline saved successfully as 'best_sentiment_pipeline.pkl'")

backend_metadata = {
    "best_model_name": best_model_name,
    "metrics": {
        "accuracy": results[best_model_name]["accuracy"],
        "precision": results[best_model_name]["precision"],
        "recall": results[best_model_name]["recall"],
        "f1_score": results[best_model_name]["f1_score"],
    },
    "confusion_matrix": results[best_model_name]["confusion_matrix"].tolist(),
    "label_mapping": label_mapping
}
joblib.dump(backend_metadata, 'backend_metadata.pkl')
print("Backend metadata saved successfully as 'backend_metadata.pkl'\n")


# ================================================
# Advanced NLP Model (BERT / Transformers)
# ================================================
print("================================================")
print("Training DistilBERT Transformer Model")
print("================================================")

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using execution device: {device}\n")

# Reformat data split for Hugging Face Dataset format
hf_train_df = pd.DataFrame({'text': X_train, 'label': y_train}).reset_index(drop=True)
hf_test_df = pd.DataFrame({'text': X_test, 'label': y_test}).reset_index(drop=True)

train_dataset = Dataset.from_pandas(hf_train_df)
test_dataset = Dataset.from_pandas(hf_test_df)

# Load pre-trained Tokenizer
transformer_model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(transformer_model_name)

# Tokenization utility function
def tokenize_inputs(data):
    return tokenizer(data["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_inputs, batched=True)
tokenized_test = test_dataset.map(tokenize_inputs, batched=True)

# Initialize Model with 3 distinct output classes
transformer_model = AutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=3)

# Define hardware-optimized training configuration
training_args = TrainingArguments(
    output_dir="./transformer_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none"
)

# Metric aggregation for Transformer evaluation
def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_transformer_metrics,
)

# Begin fine-tuning
print("Fine-tuning DistilBERT")
trainer.train()

# Evaluate the transformer performance
print("\n==== Evaluating DistilBERT Performance ====")
eval_metrics = trainer.evaluate()
print(f"DistilBERT Accuracy:  {eval_metrics['eval_accuracy']:.4f}")
print(f"DistilBERT Precision: {eval_metrics['eval_precision']:.4f}")
print(f"DistilBERT Recall:    {eval_metrics['eval_recall']:.4f}")
print(f"DistilBERT F1-Score:  {eval_metrics['eval_f1']:.4f}")

==== Loading Dataset ====
Training set size: 960
Testing set size: 240

==== Logistic Regression Performance ====
Accuracy:  0.6208
Precision: 0.5770
Recall:    0.6208
F1-Score:  0.5948

==== Naive Bayes Performance ====
Accuracy:  0.5708
Precision: 0.3259
Recall:    0.5708
F1-Score:  0.4149

🥇 Absolute Best Traditional Model: Logistic Regression (0.6208 Accuracy)
Pipeline saved successfully as 'best_sentiment_pipeline.pkl'
Backend metadata saved successfully as 'backend_metadata.pkl'

Training DistilBERT Transformer Model
Using execution device: cpu



Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


==== Evaluating DistilBERT Performance ====


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.747251,0.822848,2,0.633333,0.588855,0.633333,0.602915


DistilBERT Accuracy:  0.6333
DistilBERT Precision: 0.5889
DistilBERT Recall:    0.6333
DistilBERT F1-Score:  0.6029


In [11]:
import nltk

print("Downloading Training Dependencies")

# Core NLTK resources for tokenization and text alignment
nltk.download('punkt')
nltk.download('punkt_tab')

print("All NLTK resources downloaded")

All NLTK resources downloaded


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [13]:
import os

print("==== Generating Final Predictions CSV ====")

results_df = pd.DataFrame({
    'clean_text': X_test,
    'true_target': y_test
})

inverse_label_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
results_df['true_sentiment'] = results_df['true_target'].map(inverse_label_mapping)

for name, model in models.items():
    results_df[f'pred_{name.lower().replace(" ", "_")}'] = model.predict(X_test_tfidf)
    results_df[f'sentiment_{name.lower().replace(" ", "_")}'] = results_df[f'pred_{name.lower().replace(" ", "_")}'].map(inverse_label_mapping)

final_export_df = results_df.drop(columns=['true_target', 'pred_logistic_regression', 'pred_naive_bayes'])

drive_export_path = "/content/drive/MyDrive/dataset/final_model_predictions.csv"

os.makedirs(os.path.dirname(drive_export_path), exist_ok=True)

final_export_df.to_csv(drive_export_path, index=False)

print("saved to:")
print(f" -> {drive_export_path}")
print("\n Preview of Saved Product")
print(final_export_df.head())

==== Generating Final Predictions CSV ====
saved to:
 -> /content/drive/MyDrive/dataset/final_model_predictions.csv

 Preview of Saved Product
        clean_text true_sentiment sentiment_logistic_regression  \
1188        , ....       negative                      negative   
898   missing_text       positive                      positive   
276              .       negative                      negative   
1040         .....       negative                      negative   
554   missing_text       negative                      positive   

     sentiment_naive_bayes  
1188              positive  
898               positive  
276               positive  
1040              positive  
554               positive  
